In [3]:
import json
import pandas as pd
# 从文件中读取JSON数据
def load_data_from_json(file_path):
    with open(file_path, 'r', encoding="utf-8") as file:
        data = pd.read_json(file_path, lines=True)
    return data

# 按行读取文件中的JSON对象
def load_data_by_line(file_path):
    data = []
    with open(file_path, 'r', encoding="utf-8") as file:
        for line in file:
            # 去掉空白行
            if line.strip():
                try:
                    data.append(json.loads(line.strip()))
                except json.JSONDecodeError as e:
                    print(f"Error decoding JSON: {e}")
    return data


# 封装为训练数据格式
def process_data(data):
    training_data = []

    for item in data:
        # 确保每个样本中有需要的字段
        if all(key in item for key in ["reasonable_prompt", "reasonable_score", "reasonable_explanation"]):
            if(item["reasonable_score"] < 0.6 and item["flag"] == 1):
                continue
            if(item["reasonable_score"] > 0.6  and item["flag"] == 0):
                continue
            training_sample = {
                "input": item["reasonable_prompt"],
                "output":json.dumps({
                    "result": item["reasonable_score"],
                    "explanation": item["reasonable_explanation"]
                })
            }
            training_data.append(training_sample)
        else:
            print(f"Skipping item with missing keys: {item}")
    print("training_data 长度为",len(training_data))
    return training_data

# 保存封装好的训练数据到文件
def save_training_data_to_file(training_data, output_file_path):
    with open(output_file_path, 'w') as f:
        json.dump(training_data, f, indent=4)

# 主函数
if __name__ == "__main__":
    # 加载JSON数据
    input_file_path = "D:/PycharmProjects/TDFilter/experiment4/results.jsonl"  # 替换为你的输入文件路径
    data = load_data_by_line(input_file_path)

    # 处理数据并封装为训练数据格式
    training_data = process_data(data)

    # 保存封装后的训练数据到文件
    output_file_path = "train_confidence.json"  # 输出文件路径
    save_training_data_to_file(training_data, output_file_path)

    print(f"Training data has been saved to {output_file_path}")



training_data 长度为 7819
Training data has been saved to train_confidence.json
